# animations — TRELLIS single-image → 3D (best single-photo quality)

**TRELLIS** turns **one** photo into a detailed, **textured** 3D model — far
sharper than TripoSR. Same one-photo workflow you already use.

**Steps:** Runtime → Change runtime type → **GPU (T4 works; L4/A100 faster)**,
then Runtime → **Run all**. When the last cell prints a `gradio.live` URL, paste
it into the Studio's **Connect a generator** field and Generate with just the
**Front** photo.

> ⚠️ **Heaviest notebook by far.** It builds custom CUDA extensions and pins
> torch, so the first run takes **~15–20 min** and is the most likely to hit a
> version snag. If a cell turns red, copy the error — it's almost always one
> dependency wheel to adjust. It produces **textured** models (real colors).
> The `gradio.live` link changes every session.


In [ ]:
# 1) Install TRELLIS + its CUDA extensions on Colab (Python 3.12, CUDA 12.1).
# First run is ~15-20 min: downloads torch and COMPILES the CUDA rasterizers.
import os
os.environ["ATTN_BACKEND"] = "xformers"   # skip flash-attn (its cp312 build is painful)
os.environ["SPCONV_ALGO"] = "native"

%cd /content
!rm -rf TRELLIS
!git clone --recurse-submodules https://github.com/microsoft/TRELLIS.git
%cd /content/TRELLIS
!git submodule update --init --recursive

# We only need image-to-3D. TRELLIS's pipelines/__init__ also imports the
# TEXT pipeline, which pulls transformers -> a huggingface_hub version that
# clashes with the gradio-pinned hub 0.25.2. Drop that import.
!sed -i '/trellis_text_to_3d/d' /content/TRELLIS/trellis/pipelines/__init__.py

# Torch 2.4.0 / cu121 - has cp312 wheels and matches kaolin/xformers/spconv below.
!pip install -q torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121

# Core deps + a Studio-compatible Gradio stack (same pins as the TripoSR notebook).
!pip install -q imageio imageio-ffmpeg tqdm easydict opencv-python-headless \
    scipy ninja rembg onnxruntime "trimesh>=4.5" xatlas pyvista pymeshfix igraph \
    "numpy==2.0.2" "gradio==4.44.1" "gradio-client==1.3.0" "huggingface_hub==0.25.2" "pydantic==2.10.6"
!pip install -q git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8

# GPU building blocks - cp312 wheels for torch 2.4.0 / cu121:
!pip install -q xformers==0.0.27.post2 --index-url https://download.pytorch.org/whl/cu121
!pip install -q spconv-cu121   # cu121 ships cp312 wheels; cu120 does NOT (that was the "no versions" error)
!pip install -q kaolin==0.17.0 -f https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-2.4.0_cu121.html

# CUDA rasterizers TRELLIS renders/textures with (compiled here with nvcc).
# NOTE: mip-splatting is cloned SEPARATELY - it is NOT vendored in the TRELLIS repo,
# which is why "./extensions/.../diff-gaussian-rasterization/" did not exist.
!pip install -q git+https://github.com/NVlabs/nvdiffrast.git
!git clone --recurse-submodules https://github.com/JeffreyXiang/diffoctreerast.git /tmp/diffoctreerast && pip install -q /tmp/diffoctreerast
!git clone https://github.com/autonomousvision/mip-splatting.git /tmp/mip-splatting && pip install -q /tmp/mip-splatting/submodules/diff-gaussian-rasterization/


# Colab's pandas is built against numpy 2, but the ML stack above pulled numpy
# 1.26.4 - reinstall a numpy-1-compatible pandas so "import gradio" does not
# hit a numpy ABI mismatch (dtype size changed, 96 vs 88).
!pip install -q --force-reinstall --no-deps "pandas==2.1.4"

# Colab's cupy is built against numpy 2 and breaks under numpy 1.26.4. TRELLIS
# does not use it (it is only an optional accel inside pymatting/rembg), so
# remove it - pymatting then falls back to CPU instead of crashing on import.
!pip uninstall -y cupy-cuda12x cupy-cuda11x cupy 2>/dev/null || true

print("Install finished. If nothing above is red, run the next cell.")
print("If cell 2 errors about torch versions, do Runtime -> Restart session, then run cell 2 only.")


In [ ]:
# 2) Single-image TRELLIS app the Studio drives (api_name="generate"), + share link.
import os, tempfile
os.environ["ATTN_BACKEND"] = "xformers"
os.environ["SPCONV_ALGO"] = "native"
%cd /content/TRELLIS

import gradio as gr
from trellis.pipelines import TrellisImageTo3DPipeline
from trellis.utils import postprocessing_utils

print("Loading TRELLIS-image-large (first time downloads a few GB)...")
pipe = TrellisImageTo3DPipeline.from_pretrained("JeffreyXiang/TRELLIS-image-large")
pipe.cuda()

def generate(image):
    if image is None:
        raise gr.Error("Upload a device photo.")
    # TRELLIS removes the background itself (preprocess_image=True by default).
    outputs = pipe.run(image.convert("RGB"), seed=1, formats=["gaussian", "mesh"])
    glb = postprocessing_utils.to_glb(
        outputs["gaussian"][0], outputs["mesh"][0], simplify=0.95, texture_size=1024,
    )
    path = os.path.join(tempfile.mkdtemp(), "model.glb")
    glb.export(path)
    return path

with gr.Blocks(title="animations - TRELLIS single-image to 3D") as demo:
    gr.Markdown("## animations - TRELLIS single-image to 3D\nDriven by the Studio; you can also test here directly.")
    img = gr.Image(type="pil", label="Device photo")
    out = gr.Model3D(label="Result (.glb)")
    gr.Button("Generate", variant="primary").click(generate, img, out, api_name="generate")

print("Watch for:  Running on public URL: https://XXXX.gradio.live")
demo.queue(max_size=4).launch(share=True)


## Use it
1. Copy the `https://XXXX.gradio.live` URL printed above.
2. Open the Studio: **zurlazar.github.io/animations/studio.html**
3. Paste the URL into **Connect a generator** -> **Save**.
4. Put your photo in the **Front** slot -> **Generate 3D model** (other slots are
   ignored by this single-image model).
5. **Download .glb** when you like the result.

Quality tips: a clean, single, sharp, well-lit product shot on a plain
background still helps a lot — but TRELLIS is far more forgiving than TripoSR.
